---
## Import Required Packages

In [ ]:
import torch
import yaml
import copy
from pathlib import Path
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel

---
## Configuration Setup

In [6]:
ORIGINAL_MODEL = 'Final_solution_yolo/YOLO26-Distilled.pt'
DATA_YAML      = 'Dataset_yolo/merged_dataset/dataset.yaml'
OUTPUT_DIR     = 'Pruning/FinalFolder_28th'
PRUNED_YAML    = 'Pruning/FinalFolder_28th/yolo26s_pruned_60pct.yaml'
PRUNED_PT      = 'runs/detect/Pruning/FinalFolder_28th/finetune_60pct/weights/best.pt'
FINETUNED_PT   = 'Pruning/FinalFolder_28th/YOLO26_pruned_yaml_60pct_finetuned.pt'
DEPLOY_PT      = 'Pruning/FinalFolder_28th/YOLO26_pruned_yaml_60pct_deploy.pt'

PRUNE_RATIO    = 0.60
NC             = 4
IMGSZ          = 640
BATCH          = 16
WORKERS        = 8
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Original  : {ORIGINAL_MODEL}')
print(f'Output    : {OUTPUT_DIR}')
print(f'Device    : {DEVICE}')
print(f'Prune     : {PRUNE_RATIO:.0%} channel reduction')

Original  : Final_solution_yolo/YOLO26-Distilled.pt
Output    : Pruning/FinalFolder_28th
Device    : cuda
Prune     : 60% channel reduction


---
## Baseline validation

In [4]:
yolo_orig    = YOLO(ORIGINAL_MODEL)
results_orig = yolo_orig.val(
    data=DATA_YAML, imgsz=IMGSZ, batch=BATCH,
    conf=0.001, iou=0.6, device=DEVICE,
    verbose=False, plots=False,
)
orig_params = sum(p.numel() for p in yolo_orig.model.parameters())
orig_size   = Path(ORIGINAL_MODEL).stat().st_size / 1e6

print(f'📊  Original model')
print(f'  Params       : {orig_params:,}')
print(f'  Size         : {orig_size:.2f} MB')
print(f'  mAP@0.5      : {results_orig.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {results_orig.box.map:.4f}')

Ultralytics 8.4.24 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 15446.7±1909.4 MB/s, size: 1148.7 KB)
val: Scanning /home/ak41/Dataset_yolo/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 615.9Mit/s 0.0s


/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.2it/s 34.3s0.2s
                   all       1762       9903      0.862      0.748      0.833        0.6
Speed: 0.4ms preprocess, 3.6ms inference, 0.0ms loss, 0.3ms postprocess per image
📊  Original model
  Params       : 9,466,728
  Size         : 20.29 MB
  mAP@0.5      : 0.8332
  mAP@0.5:0.95 : 0.5997


---
## Step 5 — Build pruned yaml (60%)

In [ ]:


def prune_ch(actual_ch, ratio=0.60):
    """Compute pruned yaml channel value from actual channel count."""
    yaml_val = actual_ch * 2                    # undo width_multiple=0.5
    pruned   = int(yaml_val * (1 - ratio))      # apply ratio
    return max(8, (pruned // 8) * 8)            # round to multiple of 8

P1 = prune_ch(32,  PRUNE_RATIO)   # actual: 12  yaml: 24
P2 = prune_ch(64,  PRUNE_RATIO)   # actual: 24  yaml: 48
P3 = prune_ch(128, PRUNE_RATIO)   # actual: 48  yaml: 96
P4 = prune_ch(256, PRUNE_RATIO)   # actual: 100 yaml: 200
P5 = 1024                          # actual: 512 (kept)

print(f'Yaml channel values for {PRUNE_RATIO:.0%} pruning:')
print(f'  P1 (B0)      : yaml={P1:<5} actual={P1//2}')
print(f'  P2 (B1)      : yaml={P2:<5} actual={P2//2}')
print(f'  P3 (B2-B3)   : yaml={P3:<5} actual={P3//2}')
print(f'  P4 (B4-B6)   : yaml={P4:<5} actual={P4//2}')
print(f'  P5 (B7-B10)  : yaml={P5:<5} actual={P5//2} (kept)')

pruned_yaml = {
    'nc'      : NC,
    'channels': 3,
    'scale'   : 's',
    'end2end' : False,
    'scales'  : {
        'n': [0.5, 0.25, 1024],
        's': [0.5, 0.5,  1024],
        'm': [0.5, 1.0,  512 ],
        'l': [1.0, 1.0,  512 ],
        'x': [1.0, 1.5,  512 ],
    },
    'backbone': [
        [-1, 1, 'Conv',  [P1,  3, 2]],
        [-1, 1, 'Conv',  [P2,  3, 2]],
        [-1, 2, 'C3k2', [P3,  False, 0.25]],
        [-1, 1, 'Conv',  [P3,  3, 2]],
        [-1, 2, 'C3k2', [P4,  False, 0.25]],
        [-1, 1, 'Conv',  [P4,  3, 2]],
        [-1, 2, 'C3k2', [P4,  True]],
        [-1, 1, 'Conv',  [P5,  3, 2]],
        [-1, 2, 'C3k2', [P5,  True]],
        [-1, 1, 'SPPF', [P5,  5, 3, True]],
        [-1, 2, 'C2PSA',[P5]],
    ],
    'head': [
        [-1,       1, 'nn.Upsample', ['None', 2, 'nearest']],
        [[-1, 6],  1, 'Concat',      [1]],
        [-1,       2, 'C3k2',        [P4, True]],
        [-1,       1, 'nn.Upsample', ['None', 2, 'nearest']],
        [[-1, 4],  1, 'Concat',      [1]],
        [-1,       2, 'C3k2',        [P3, True]],
        [-1,       1, 'Conv',        [P3, 3, 2]],
        [[-1, 13], 1, 'Concat',      [1]],
        [-1,       2, 'C3k2',        [P4, True]],
        [-1,       1, 'Conv',        [P4, 3, 2]],
        [[-1, 10], 1, 'Concat',      [1]],
        [-1,       1, 'C3k2',        [P5, True, 0.5, True]],
        [[16,19,22], 1, 'Detect',    ['nc']],
    ],
}

with open(PRUNED_YAML, 'w') as f:
    yaml.dump(pruned_yaml, f, default_flow_style=False)
print(f'\n✓ Pruned yaml saved: {PRUNED_YAML}')

Yaml channel values for 60% pruning:
  P1 (B0)      : yaml=24    actual=12
  P2 (B1)      : yaml=48    actual=24
  P3 (B2-B3)   : yaml=96    actual=48
  P4 (B4-B6)   : yaml=200   actual=100
  P5 (B7-B10)  : yaml=1024  actual=512 (kept)

✓ Pruned yaml saved: Pruning/FinalFolder_28th/yolo26s_pruned_60pct.yaml


---
## Build pruned model from yaml

In [ ]:

pruned_model = DetectionModel(PRUNED_YAML, nc=NC)
pruned_model.eval()

pruned_params = sum(p.numel() for p in pruned_model.parameters())
print(f'\n✓ Pruned model built')
print(f'  Params    : {pruned_params:,}')
print(f'  Reduction : {100*(orig_params-pruned_params)/orig_params:.1f}%')
print(f'  Expected  : ~60% reduction → ~{orig_params*0.4/1e6:.2f}M params')

Building pruned model from yaml ...

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      3504  ultralytics.nn.modules.conv.Conv             [16, 24, 3, 2]                
  2                  -1  1      3780  ultralytics.nn.modules.block.C3k2            [24, 48, 1, False, 0.25]      
  3                  -1  1     20832  ultralytics.nn.modules.conv.Conv             [48, 48, 3, 2]                
  4                  -1  1     17082  ultralytics.nn.modules.block.C3k2            [48, 104, 1, False, 0.25]     
  5                  -1  1     97552  ultralytics.nn.modules.conv.Conv             [104, 104, 3, 2]              
  6                  -1  1     57616  ultralytics.nn.modules.block.C3k2            [104, 104, 1, True]           
  7                  -1  1    480256  ultralytics.n

---
## Step 7 — Transfer weights

In [ ]:

ckpt      = torch.load(ORIGINAL_MODEL, map_location='cpu', weights_only=False)
src_model = ckpt.get('model') or ckpt.get('ema') or ckpt
src_sd    = src_model.state_dict()
dst_sd    = pruned_model.state_dict()

transferred    = 0
skipped        = 0
shape_mismatch = 0
head_skipped   = 0

for k in dst_sd:
    # Skip Detect head — reg_max changed from 1 to 16
    if k.startswith('model.23.'):
        head_skipped += 1
        continue
    if k not in src_sd:
        skipped += 1
        continue
    s = src_sd[k]
    d = dst_sd[k]
    if s.shape == d.shape:
        dst_sd[k] = s
        transferred += 1
    elif s.dim() == d.dim():
        slices = tuple(slice(0, min(s.shape[i], d.shape[i])) for i in range(s.dim()))
        dst_sd[k] = s[slices]
        transferred += 1
        shape_mismatch += 1
    else:
        skipped += 1

pruned_model.load_state_dict(dst_sd, strict=False)
print(f'✓ Weight transfer complete')
print(f'  Exact match    : {transferred - shape_mismatch}')
print(f'  Sliced to fit  : {shape_mismatch}')
print(f'  Head skipped   : {head_skipped} (Detect head — random init)')
print(f'  Skipped        : {skipped}')
del ckpt, src_model, src_sd, dst_sd

Loading original model weights ...
✓ Weight transfer complete
  Exact match    : 216
  Sliced to fit  : 252
  Head skipped   : 121 (Detect head — random init)
  Skipped        : 0


---
##  Verify forward pass

In [ ]:

pruned_model.eval()
dummy = torch.randn(1, 3, IMGSZ, IMGSZ)
try:
    with torch.no_grad():
        out = pruned_model(dummy)
    print('Forward pass OK')
    if isinstance(out, (list, tuple)):
        print(f'  Output length : {len(out)}')
        for i, o in enumerate(out):
            if hasattr(o, 'shape'):
                print(f'    out[{i}] : {o.shape}')
except Exception as e:
    print(f'Forward pass failed: {e}')

Testing forward pass ...
✅ Forward pass OK
  Output length : 2
    out[0] : torch.Size([1, 8, 8400])


---
## Save pruned model

In [9]:
pruned_model.eval()
pruned_model_half = copy.deepcopy(pruned_model).half()

ckpt_orig = torch.load(ORIGINAL_MODEL, map_location='cpu', weights_only=False)
ckpt_save = {
    'model'       : pruned_model_half,
    'epoch'       : -1,
    'best_fitness': None,
    'optimizer'   : None,
    'train_args'  : ckpt_orig.get('train_args', {}),
    'date'        : None,
    'version'     : None,
}
torch.save(ckpt_save, PRUNED_PT)
del ckpt_orig

size_mb = Path(PRUNED_PT).stat().st_size / 1e6
print(f'✓ Pruned model saved: {PRUNED_PT}')
print(f'  Params    : {pruned_params:,}')
print(f'  Size      : {size_mb:.2f} MB')
print(f'  Reduction : {100*(orig_params-pruned_params)/orig_params:.1f}%')

# Verify end2end is False
ckpt_v = torch.load(PRUNED_PT, map_location='cpu', weights_only=False)
m_v    = ckpt_v.get('model') or ckpt_v
print(f'  end2end   : {getattr(m_v.model[-1], "end2end", "not present")}  ← must be False')
del ckpt_v, m_v, pruned_model_half

✓ Pruned model saved: Pruning/FinalFolder_28th/YOLO26_pruned_yaml_60pct.pt
  Params    : 6,342,370
  Size      : 13.06 MB
  Reduction : 33.0%
  end2end   : False  ← must be False


---
## Validate pruned model (before fine-tuning)

In [10]:
print('Validating pruned model before fine-tuning ...')
yolo_pruned    = YOLO(PRUNED_PT)
results_pruned = yolo_pruned.val(
    data=DATA_YAML, imgsz=IMGSZ, batch=BATCH,
    conf=0.001, iou=0.6, device=DEVICE,
    verbose=False, plots=False,
)
print(f'\n📊  After pruning (before fine-tune)')
print(f'  mAP@0.5      : {results_pruned.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {results_pruned.box.map:.4f}')
print(f'  mAP drop     : {results_pruned.box.map50 - results_orig.box.map50:+.4f}')

Validating pruned model before fine-tuning ...
Ultralytics 8.4.24 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_60pct summary (fused): 120 layers, 6,329,477 parameters, 13,068 gradients, 8.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 17658.4±2909.5 MB/s, size: 1376.0 KB)
val: Scanning /home/ak41/Dataset_yolo/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 671.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.1it/s 36.2s0.2s
                   all       1762       9903          0          0          0          0
Speed: 0.3ms preprocess, 3.7ms inference, 0.0ms loss, 3.0ms postprocess per image

📊  After pruning (before fine-tune)
  mAP@0.5      : 0.0000
  mAP@0.5:0.95 : 0.0000
  mAP drop     : -0.8332


---
## Fine-tune

In [ ]:
print('Starting fine-tuning ...')

yolo_ft = YOLO('runs/detect/Pruning/FinalFolder_28th/finetune_60pct/weights/best.pt')

results_ft = yolo_ft.train(
    data          = DATA_YAML,
    imgsz         = IMGSZ,
    batch         = BATCH,
    workers       = WORKERS,
    epochs        = 80,
    patience      = 30,
    warmup_epochs = 5,
    lr0           = 1e-4,
    lrf           = 0.01,
    optimizer     = 'AdamW',
    weight_decay  = 1e-4,
    mosaic        = 1.0,
    scale         = 0.9,
    fliplr        = 0.5,
    degrees       = 5.0,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    copy_paste    = 0.3,
    amp           = False,
    freeze        = 0,
    project       = OUTPUT_DIR,
    name          = 'finetune_60pct',
    device        = DEVICE,
    save          = True,
    save_period   = 5,
    plots         = True,
    verbose       = True,
)

best_ft = Path(OUTPUT_DIR) / 'finetune_60pct' / 'weights' / 'best.pt'
print('Fine-tuning complete')
print(f'  Best: {best_ft}')

Starting fine-tuning ...
New https://pypi.org/project/ultralytics/8.4.31 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.24 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Dataset_yolo/merged_dataset/dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, m

/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7726.0±7235.8 MB/s, size: 1063.3 KB)
val: Scanning /home/ak41/Dataset_yolo/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 246.3Mit/s 0.0s


/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 96 weight(decay=0.0), 103 weight(decay=0.0001), 102 bias(decay=0.0)
Plotting labels to /home/ak41/runs/detect/Pruning/FinalFolder_28th/finetune_60pct2/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/ak41/runs/detect/Pruning/FinalFolder_28th/finetune_60pct2
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/80      4.49G      2.082      1.576      1.283        191        640: 100% ━━━━━━━━━━━━ 514/514 2.1it/s 4:01<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.0it/s 27.4s0.8s
                   all       1762       9903      0.375      0.275      0.278      0.131

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/80      4.51G      2.036      1.516      1.261        129        640: 100% ━━━━━━

/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      71/80      4.71G      1.444      1.015      1.049         66        640: 100% ━━━━━━━━━━━━ 514/514 2.4it/s 3:32<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.0it/s 27.5s0.7ss
                   all       1762       9903      0.729      0.562      0.642      0.364

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      72/80      4.71G      1.426     0.9857      1.044         68        640: 100% ━━━━━━━━━━━━ 514/514 2.5it/s 3:26<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.1it/s 26.3s0.7s
                   all       1762       9903      0.729      0.551      0.638      0.363

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      73/80      4.71G       1.41     0.9795      1.038         32        

---
## Validate fine-tuned model

In [13]:
from pathlib import Path
CLASS_NAMES = ['pedestrian', 'cyclist', 'car', 'large_vehicle']

best_ft   = Path('runs/detect/Pruning/FinalFolder_28th/finetune_60pct2/weights/best.pt')
yolo_best = YOLO(str(best_ft))

results_ft_val = yolo_best.val(
    data=DATA_YAML, imgsz=IMGSZ, batch=BATCH,
    conf=0.001, iou=0.6, device=DEVICE,
    verbose=False, plots=True,
    project=OUTPUT_DIR, name='finetune_60pct_val',
)

print('\n' + '═'*65)
print(f"  {'Metric':<25} {'Original':>12} {'Fine-tuned':>12}")
print('═'*65)
print(f"  {'mAP@0.5':<25} {'0.8332':>12} {results_ft_val.box.map50:>12.4f}")
print(f"  {'mAP@0.5:0.95':<25} {'0.5997':>12} {results_ft_val.box.map:>12.4f}")
print(f"  {'Precision':<25} {'0.8623':>12} {results_ft_val.box.mp:>12.4f}")
print(f"  {'Recall':<25} {'0.7480':>12} {results_ft_val.box.mr:>12.4f}")
print('─'*65)
print(f"  {'File size (MB)':<25} {'20.29':>12} {best_ft.stat().st_size/1e6:>12.2f}")
print('═'*65)
print('\nPer-class mAP@0.5:')
for i, name in enumerate(CLASS_NAMES):
    try:
        print(f'  {name:<20} {results_ft_val.box.ap50[i]:.4f}')
    except:
        pass

Ultralytics 8.4.24 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_60pct summary (fused): 120 layers, 6,329,477 parameters, 0 gradients, 8.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 16774.8±1910.4 MB/s, size: 1695.1 KB)
val: Scanning /home/ak41/Dataset_yolo/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 615.9Mit/s 0.0s


/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/ak41/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 36.5s0.3s
                   all       1762       9903      0.774       0.58      0.668      0.372
Speed: 0.2ms preprocess, 3.2ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /home/ak41/runs/detect/Pruning/FinalFolder_28th/finetune_60pct_val3

═════════════════════════════════════════════════════════════════
  Metric                        Original   Fine-tuned
═════════════════════════════════════════════════════════════════
  mAP@0.5                         0.8332       0.6677
  mAP@0.5:0.95                    0.5997       0.3720
  Precision                       0.8623       0.7738
  Recall                          0.7480       0.5797
─────────────────────────────────────────────────────────────────
  File size (MB)                   20.29        13.02
═════════════════════════════════════════════════════════════════

Per-class mAP@0.5:
 

---
## Save Fine Tuned model

In [14]:
import shutil

best_ft = 'runs/detect/Pruning/FinalFolder_28th/finetune_60pct2/weights/best.pt'

# Strip EMA + optimizer — keep weights only
ckpt  = torch.load(str(best_ft), map_location='cpu', weights_only=False)
model = ckpt.get('ema') or ckpt.get('model')
torch.save({'model': model.half()}, DEPLOY_PT)

deploy_size  = Path(DEPLOY_PT).stat().st_size / 1e6
pruned_size  = Path(PRUNED_PT).stat().st_size / 1e6

print(f'✓ Deployment model saved : {DEPLOY_PT}')
print(f'  Pruned (before FT) : {pruned_size:.2f} MB')
print(f'  Deployment (after) : {deploy_size:.2f} MB')
print(f'  Original           : {orig_size:.2f} MB')
print(f'  Size reduction     : {100*(orig_size-deploy_size)/orig_size:.1f}%')
print(f'\n  All outputs in     : {OUTPUT_DIR}/')

✓ Deployment model saved : Pruning/FinalFolder_28th/YOLO26_pruned_yaml_60pct_deploy.pt
  Pruned (before FT) : 38.65 MB
  Deployment (after) : 13.04 MB
  Original           : 20.29 MB
  Size reduction     : 35.7%

  All outputs in     : Pruning/FinalFolder_28th/
